<a href="https://colab.research.google.com/github/CenkAydin/TDL-ADD/blob/main/TDL_Mamba_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TDL-ADD — Aşama 2: Mamba Eğitimi (Colab)
**Hücre çalıştırma sırası:** 1 → 2 → 3 → 4  
Hücre 3 uzun sürer (~30–60 dk preprocess). Runtime yeniden başlatılırsa sadece Hücre 4'ü çalıştırabilirsiniz (özellikler `/content/` diskinde duruyorsa).

In [1]:
# ── Hücre 1: Drive Bağlama + Repo Klonlama ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Checkpoint ve score çıktıları için Drive klasörlerini önceden oluştur
import os
os.makedirs('/content/drive/MyDrive/TDL-ADD/models/A2_Mamba/checkpoints_A2_Mamba', exist_ok=True)
os.makedirs('/content/drive/MyDrive/TDL-ADD/scores/A2_Mamba', exist_ok=True)

# Repo klonla (zaten varsa atla)
if not os.path.exists('/content/TDL-ADD'):
    !git clone https://github.com/CenkAydin/TDL-ADD /content/TDL-ADD
else:
    print('Repo zaten mevcut, güncelleniyor...')
    !git -C /content/TDL-ADD pull

%cd /content/TDL-ADD
print('Çalışma dizini:', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo zaten mevcut, güncelleniyor...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.86 KiB | 1.86 MiB/s, done.
From https://github.com/CenkAydin/TDL-ADD
   e897652..6c2ceef  main       -> origin/main
Updating e897652..6c2ceef
Fast-forward
 TDL_Mamba_Colab.ipynb | 278 ++++++++++++--------------------------------------
 1 file changed, 67 insertions(+), 211 deletions(-)
/content/TDL-ADD
Çalışma dizini: /content/TDL-ADD


In [2]:
# ── Hücre 2: Akıllı Mamba Kurulumu (Drive Caching) ────────────────────────
import os
import glob

os.environ['CUDA_HOME']                 = '/usr/local/cuda'
os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
os.environ['MAMBA_FORCE_BUILD']         = 'TRUE'
os.environ['MAX_JOBS']                  = '4'

!nvcc --version
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)"
!pip install packaging ninja --quiet

WHEEL_DIR = '/content/drive/MyDrive/TDL-ADD/wheels'
os.makedirs(WHEEL_DIR, exist_ok=True)

# Drive'da daha önceden kaydettiğimiz wheel (.whl) dosyaları var mı kontrol et
cached_wheels = glob.glob(f"{WHEEL_DIR}/*.whl")

if len(cached_wheels) >= 2:
    print("\n🚀 Drive'da hazır derlenmiş Mamba dosyaları bulundu! 5 saniyede kuruluyor...")
    !pip install {WHEEL_DIR}/*.whl --no-index
else:
    print("\n⏳ Drive'da hazır dosya yok. Mamba sıfırdan derleniyor (Bu biraz zaman alacak)...")
    !pip install causal-conv1d --no-build-isolation
    !pip install mamba-ssm --no-build-isolation

    print("\n💾 Derleme bitti! Bir sonraki Colab çökmesinde saniyeler içinde kurabilmek için Drive'a yedekleniyor...")
    # Ajanın patlayan cp kodu yerine, find komutu ile iç içe klasörlerdeki tüm whl dosyalarını bulup kopyalıyoruz
    !find /root/.cache/pip/wheels/ -name "*.whl" -exec cp {} {WHEEL_DIR}/ \;
    print("✅ Tekerlekler (Wheels) Drive'a başarıyla kaydedildi!")

!python -c "from mamba_ssm import Mamba; print('\nmamba-ssm OK 🚀')"
!pip install transformers tqdm pytorch-model-summary --quiet

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
PyTorch: 2.9.0+cu126 | CUDA: 12.6

⏳ Drive'da hazır dosya yok. Mamba sıfırdan derleniyor (Bu biraz zaman alacak)...
  Using cached causal_conv1d-1.6.1.tar.gz (29 kB)
  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl size=193269711 sha256=0698e41cbeb31e684abb2308a20ba870dd9b8e5475d25194f5fdfc62b97c2997
  Stored in directory: /root/.cache/pip/wheels/98/4a/75/b24971cff4599825b16b612f08fbd2e60a2c336a56e081a3c8
Successfully built causal-conv1d
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 17.6 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=370557372 sha256=b3ef8cff54058766a2227048

In [ ]:
# ── Hücre 3: Veri İndirme + Açma + Preprocess (DÜZELTİLMİŞ KUSURSUZ VERSİYON) ──
import os, glob

BASE_DIR     = '/content/asv2019PS'
DATA_ROOT    = '/content/asv2019PS/database'
FEATURE_ROOT = '/content/asv2019PS/preprocess_A1_WavLM_Large'
ARCHIVE_PATH = '/content/asv2019ps_archive.zip'
RAW_DIR      = '/content/asv2019PS_raw'

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

# 1) Zenodo arşivini indir (zaten varsa atla)
if not os.path.exists(ARCHIVE_PATH):
    print('Zenodo arşivi indiriliyor...')
    !wget -q --show-progress 'https://zenodo.org/api/records/5766198/files-archive' -O {ARCHIVE_PATH}
else:
    print('Arşiv mevcut, indirme atlandı. ✅')

# 2) Dış zip'i aç (zaten varsa atla)
if not os.path.exists(RAW_DIR):
    print('Zip açılıyor...')
    !unzip -q {ARCHIVE_PATH} -d {RAW_DIR}
else:
    print('Zip zaten açılmış. ✅')

# 3) Tar.gz dosyalarını çıkartma
if not os.path.exists(os.path.join(DATA_ROOT, 'train')) and not os.path.exists(os.path.join(DATA_ROOT, 'database', 'train')):
    tar_files = sorted(glob.glob(f'{RAW_DIR}/*.tar.gz'))
    print(f'\n{len(tar_files)} adet tar.gz bulundu, çıkartılıyor...')
    for f in tar_files:
        print(f'  → {os.path.basename(f)}')
        # HATA BURADAYDI: -C BASE_DIR yapıyoruz ki database/database olmasın!
        !tar -xzf {f} -C {BASE_DIR}/
else:
    print('Tar.gz dosyaları zaten çıkartılmış. ✅')

# 🛠️ KRİTİK KURTARMA OPERASYONU: Eski hatalı çıkarma yüzünden oluşan iç içe klasörü düzelt
if os.path.exists(os.path.join(DATA_ROOT, 'database', 'train')):
    print('\nİç içe klasör (database/database) tespit edildi! 1 saniyede düzeltiliyor... 🛠️')
    !mv {DATA_ROOT}/database/* {DATA_ROOT}/
    !rm -rf {DATA_ROOT}/database
    print('Klasör yapısı başarıyla onarıldı! ✅')

print('\nDatabase içeriği (train, dev, eval görünmeli):')
!ls {DATA_ROOT}

# 4) WavLM-Large ile özellik çıkartma
if not os.path.exists(os.path.join(FEATURE_ROOT, 'train', 'wavlm-large')):
    print('\n🔥 Özellik çıkartma (Preprocess) başlıyor... Arkana yaslan!')
    !python /content/TDL-ADD/preprocess.py \
        --database_dir {DATA_ROOT} \
        --protocol_dir /content/TDL-ADD/label \
        --output_dir   {FEATURE_ROOT}
else:
    print('Özellikler zaten çıkartılmış. ✅')

Zenodo arşivi indiriliyor...
/content/asv2019ps_     [     <=>            ]   8.17G  2.67MB/s    in 60m 34s 


In [ ]:
# ── Hücre 4: Eğitimi Başlat ───────────────────────────────────────────────────
# Çıktılar: checkpointler ve loglar doğrudan Drive'a yazılır.
!python /content/TDL-ADD/main_train.py \
    -m  TDL_Mamba \
    -f  /content/asv2019PS/preprocess_A1_WavLM_Large \
    -d  /content/asv2019PS/database \
    -o  /content/drive/MyDrive/TDL-ADD/models/A2_Mamba \
    --ckpt_subdir checkpoints_A2_Mamba \
    --num_epochs  200 \
    --batch_size  24 \
    --lr          0.0001 \
    --lam         0.1 \
    --num_workers 2 \
    --base_loss   bce \
    --gpu         0